# Counting ships in the port of Constanța with Sentinel-1
*Luca Tarchila, Paulian Oprescu, Phuong Nguyen, Ravian Hendriks · Geoscripting, WUR, 2026*

The Black Sea region is one of the world's most important grain exporting regions: before 2022, Russia and Ukraine together supplied around a third of global wheat exports. The Russian invasion of Ukraine in February 2022 blocked Ukrainian ports, and after Russia left the Black Sea Grain Initiative in July 2023, much of Ukraine's grain was rerouted through Romanian ports. Constanța, the largest port on the Black Sea, became a key alternative route, which we expect to show up as stronger fluctuations in vessel traffic.

We count vessels in the port and the offshore anchorage on Sentinel-1 radar images from 2019 to 2025, using Google Earth Engine. Radar works through clouds and at night. Calm sea water reflects the radar signal away from the satellite (low backscatter), while a ship's metal hull and deck reflect it straight back (high backscatter), so ships stand out clearly against the water. We compare our counts with IMF PortWatch port calls, which are based on ship transponder (AIS) data.

All port-specific settings are stored in one `Port` object (`functions/detection.py`), so the same workflow can be repeated for another port by making a new instance.

**Questions**
1. Can we build a simple vessel detection algorithm based on a backscatter (dB) threshold that separates ships from the dark sea water?
2. Does our detection algorithm capture the trend in vessel traffic at Constanța, as recorded by IMF PortWatch, during the conflict in the Black Sea region?

**Table of contents**
* **1. Setup:** Pixi environment, imports and Google Earth Engine access
* **2. Settings:** the `constanta` port object and all analysis settings in one cell
* **3. Study area and data:** port and anchorage zones, datasets, images per year
* **4. Sentinel-1 images:** why we use the VH band
* **5. Ship detection on one scene:** fixed threshold vs CFAR
* **6. Time series:** ship counts in every scene, saved to `data/`
* **7. Before and after the war:** comparison with IMF PortWatch
* **8. Timelapse:** detected ships around Russia leaving the grain deal (July 2023)
* **9. Limitations**
* **10. Conclusion**
* **11. With more time**

## 1. Setup

### 1.1 Pixi environment
All packages for this project are listed in `pixi.toml`. To install them, open a terminal in the project folder and run:

    pixi install

Then select the Pixi interpreter as the notebook kernel (top right in Positron: **Python (Pixi: default)**). After that, all imports below should work.

### 1.2 Imports

In [2]:
import os
import ee
import geemap
import pandas as pd
from functions.detection import Port
import matplotlib.pyplot as plt

### 1.3 Before you start: Google Earth Engine access
https://geoscripting-wur.github.io/Python-GEE/

This project uses Google Earth Engine. To work with Google Earth Engine (GEE), you will need a Google account for authorization. The authentication process can be initiated by calling the Authenticate function from the ee module. Follow these steps:

1. After running the Authenticate function below, a URL will be printed in the output.
2. Click on the URL, which will redirect you to a Google account login page.
3. Log in to your Google account to create an authorization token.
4. When prompted with a warning stating that Google has not verified the app, click “Continue”. This warning is shown because the code we are about to run is not verified by Google, ensuring your account’s security.
5. On the authorization page, select all the checkboxes. This grants permission to access GEE data and manage data and permissions in the cloud storage where the analysis will be performed.
6. Finally, you will see an authorization code. Copy the code and paste it into the text box that appears after running the Authenticate function. Press Enter to complete the authentication process.

In [3]:
# Log in to Google Earth Engine (only need to do it once per environment)
ee.Authenticate()

True

### 1.4 Connect to your Google Earth Engine project
After logging in, connect to your own Earth Engine project. Replace the project ID below with yours.

Don't have a project yet? Create one at https://code.earthengine.google.com/register and use its ID here. Each person running this notebook uses their own project, which is why it is set here and not in the Settings.

In [4]:
# Plug in your own Google Earth Engine project ID (see 'Before you start')
gee_project = 'tutorial-geoscripting-paulian'

ee.Initialize(project=gee_project)

/home/osboxes/endproject/project-starter-furry_heavenly_elk/.pixi/envs/default/lib/python3.12/site-packages/ee/data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(
*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


## 2. Settings
All choices for the analysis are in this cell. Change a value here and run the notebook again. Nothing further down needs editing.

The settings are split in two:

**Port settings** describe the study area and are stored in a `Port` object (defined in `functions/detection.py`). Here we create one instance of this class for Constanța: `constanta`. It holds the bounding box, the longitude that splits port and anchorage, the Sentinel-1 orbit and the water shapefile. All methods (`load_water`, `load_s1`, `detect_ships`, ...) read these values from the object itself, so they don't have to be passed every time.

**Analysis settings** are choices you can tune, like the dates and the detection threshold. They are passed to the methods when they are called, so you can see in each cell which settings are used.

**Another port?** Because everything port-specific lives in one object, the workflow can be repeated for another port by making a new instance, for example:

    odesa = Port(
        name='Odesa',
        bbox=[west, south, east, north],
        split_lon=...,                  # longitude between port and anchorage
        orbit=...,                      # relative orbit that covers the port most often
        water_shp='masks/odesa_water.shp',
    )

and using `odesa` instead of `constanta` in the cells below.

In [5]:
# Specifying Constanta (RO) Port object class instance
constanta = Port(
    name='Constanta',                  # Study area
    bbox=[28.64, 44.08, 28.88, 44.18], # Bounding box [west, south, east, north] in degrees (WGS84)
                                       # Covers Constanța port and the offshore anchorage
    split_lon=28.70,                   # Longitude that split the box: port to the west, anchorage to the east
    orbit=58,                          # Sentinel-1 relative orbit 58 covers the area most often. Using one orbit
                                       # keeps the viewing angle the same in every image.
    water_shp='masks/clip_water.shp',
)

# Analysis settings (you can choose/tune these as you want)
start_date = '2019-01-01'
end_date = '2026-01-01'
threshold = -15     # dB: water pixels brighter than this are ship pixels
min_pixels = 8      # smallest group of bright pixels that counts as a ship
water_buffer = 10   # m: shrink the water mask inward to 
data_dir = 'data'
output_dir = 'output'

# Methods we tried (see 'What we tried')
k = 4                  # adaptive and CFAR: standard deviations above the background
guard = 20             # CFAR: radius around the pixel that is ignored
outer = 25             # Cfar: outer radius of the background ring, in pixels

# Create the data and output folders if they don't exist yet
os.makedirs(data_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

## 3. Study area and data
### 3.1 Study area
The Port of Constanța (Romania) is the largest port on the Black Sea. After the Russian invasion of Ukraine in February 2022 blocked shipping from Ukrainian ports, Constanța became one of the main alternative routes for Ukrainian grain exports. This makes it a good place to see whether the war changed shipping traffic.

We count ships in two zones (Figure 1):
- **Port**: the harbour basins and quays where ships load and unload.
- **Anchorage**: the offshore area where ships wait for a free berth.

We keep the zones seperate because they tell different things: ships in the port are already (un)loading their freight, while ships at anchor are waiting, so a rise in anchored ships can mean there is a congestion. Detection is also harder inside the port, where docks, cranes and moored ships sit close together, than on the open sea. 

We only look at pixels on water. To do this we clip an OpenStreetMap water polygon to the study area. Without it, cranes, containers and buildings on land would reflect the radar strongly and be counted as ships.


In [6]:
# https://geemap.org/cheatsheet/#add-earth-engine-layers
# https://geoscripting-wur.github.io/Python-GEE/

# Water mask from the OSM shapefile and bbox stored in constanta,
# shrunk inward by water_buffer (10 m, see Settings) so quay edges are not counted as ships
water = constanta.load_water(buffer=water_buffer)

# Split at constanta.split_lon (28.70°E): port to the west, anchorage to the east
west, south, east, north = constanta.bbox
port = ee.Geometry.Rectangle([west, south, constanta.split_lon, north])
anchorage = ee.Geometry.Rectangle([constanta.split_lon, south, east, north])

# Show both zones on a map (Figure 1)
m = geemap.Map()
m.centerObject(water, 12)
m.addLayer(port, {'color': 'blue'}, 'port')
m.addLayer(anchorage, {'color': 'red'}, 'anchorage')
m

Map(center=[44.12994105458296, 28.76675569965496], controls=(WidgetControl(options=['position', 'transparent_b…

### 3.2 Data

In this project we use three datasets.

| Dataset | Source | Resolution | Period | Used for |
|---|---|---|---|---|
| Sentinel-1 GRD (`COPERNICUS/S1_GRD`) | ESA Copernicus, via Google Earth Engine | 10 m, every 6 to 12 days | 2019–2025 | Data to make vessel detection predictor |
| Water polygons | OpenStreetMap | vector | – | water mask |
| Daily port calls (port260) | IMF PortWatch, based on ship transponder (AIS) data | per day | 2019–2025 | Comparing prediction to port calls |

Sentinel-1
Water Polygons
Daily port calls

**Why radar?**
Sentinel-1 bands send out radar wave signals, so it works through clouds and at night. Calm water also reflects the signal away from the satellite and it looks way darker than other objects. A ship's deck bounces it straight back to the satellite and looks like bright pixels. This makes it easy to seperate from the sea.

**Which bands**
We use VV polarisation, and one orbit direction, so every image looks at the port from the same angle. 

**Image availability**
Figure 2 shows how many images we have per year. Sentinel-1B stopped working in December 2021, so from 2022 onwards there are fewer images. The fewer images make the yearly count less certain, which we will note in the Limitations.

In [7]:
import plotly.express as px

# Study area and orbit come from the constanta instance
study_area = ee.Geometry.Rectangle(constanta.bbox)

images = (ee.ImageCollection("COPERNICUS/S1_GRD")
          .filterBounds(study_area)
          .filterDate(start_date, end_date)
          .filter(ee.Filter.eq("relativeOrbitNumber_start", constanta.orbit)))

# Number of images per year
dates = images.aggregate_array("system:time_start").getInfo()
years = pd.to_datetime(dates, unit="ms").year
counts = pd.Series(years).value_counts().sort_index()

fig = px.bar(x=counts.index, y=counts.values,
             labels={"x": "Year", "y": "Number of images"},
             title="Sentinel-1 images per year")
fig.show()

## 4. Sentinel-1 images
Sentinel-1 is a radar satellite from ESA. It sends out microwave signals and measures how much bounces back (in dB), so it works through clouds and at night. It measures two polaristions: 

- **VV**: sent vertical, received vertical.
- **VH**: sent vertical, received horizontal.

**The sea**: calm water reflects the signal away, so it looks dark. Waves make it brighter in VV, but in VH the sea stays very dark.

**Ships**: ships and cranes bounce the signal straight back, so ships are bright in both.

We use VH because the darker sea gives more contrast between ships and water (Figure 4). The trade-off is that VH is a weaker signal, so smaller boats can be missed.


In [8]:
# https://geemap.org/notebooks/04_split_panel_map/

# Study area from the constanta instance
aoi = ee.Geometry.Rectangle(constanta.bbox)

# All Sentinel-1 scenes over Constanța in orbit 58 (constanta.orbit), 2019 to 2025
s1 = constanta.load_s1(start_date, end_date)
print('Number of Sentinel-1 scenes:', s1.size().getInfo())

# First scene, to compare VH (left) and VV (right) side by side (Figure 4)
image_vv_vh = s1.first().clip(aoi)

left_layer = geemap.ee_tile_layer(image_vv_vh.select('VH'), {"min": -25, "max": 0})
right_layer = geemap.ee_tile_layer(image_vv_vh.select('VV'), {"min": -25, "max": 0})

Map = geemap.Map()
Map.centerObject(aoi, 12)
Map.split_map(left_layer, right_layer)
Map

Number of Sentinel-1 scenes: 322


Map(center=[44.13004869355397, 28.76000000000085], controls=(ZoomControl(options=['position', 'zoom_in_text', …

## 5. Ship detection on one scene
To recognize ships in an image, we are testing two methods to find them.
1. **Fixed threshold:** every water pixel brighter than 'threshold' is a ship pixel
2. **CFAR (Constant False Alarm Rate):** a pixel is a ship if it is much brighter than the water around it

Touching bright pixels will form one object, and objects smaller than min_pixels are seen as noise and will be removed.

Due to a change in ESA Sentinel 1 satellite from 2022-2025, the scatter contrast values changed, and the CFAR algorithm is unreliable. Therefore the **Fixed Threshold** method is used which detects candidate vessel cells if they are brighter than -15 dB and have a minimum of 8 candidate pixels next to them, hence filtering for bigger objects.

In [9]:
# Detect ships on the first scene with the fixed threshold (-15 dB, min 8 pixels, see Settings)
image = s1.first()
print('Date:', image.date().format('YYYY-MM-dd HH:mm').getInfo())
ships = constanta.detect_ships(image, water, method='threshold', threshold=threshold, min_pixels=min_pixels)
# ships_cfar = constanta.detect_ships(image, water, method='cfar', k=k, guard=guard, outer=outer, min_pixels=min_pixels)

# Split the detections into the two zones
ships_port = ships.filterBounds(port)
ships_anchorage = ships.filterBounds(anchorage)
print('Total:', ships.size().getInfo(), '| Port:', ships_port.size().getInfo(), '| Anchorage:', ships_anchorage.size().getInfo())

# Show the water mask and detected ships per zone
m = geemap.Map()
m.centerObject(water, 12)
m.addLayer(water, {'color': 'blue'}, 'water mask')
m.addLayer(ships_port, {'color': 'red'}, 'ships - port')
m.addLayer(ships_anchorage, {'color': 'orange'}, 'ships - anchorage')
# m.addLayer(ships_cfar, {'color': 'yellow'}, 'ships - CFAR')
m

Date: 2019-01-04 16:00
Total: 152 | Port: 147 | Anchorage: 5


Map(center=[44.12994105458296, 28.76675569965496], controls=(WidgetControl(options=['position', 'transparent_b…

## 6. Time series
We count the ships in every scence from 2019 to 2025.
This process will take a few minutes, so the result is saved to 'data/ship_counts.csv. 
The next time you run the notebook, the counts are loaded from that file instead.

In [ ]:
# Count ships in every scene, separately for port and anchorage (fixed threshold, see Settings)
# counts_cfar = constanta.ship_counts(s1, water, method='cfar', guard=guard, outer=outer, k=k, min_pixels=min_pixels)
counts_port = constanta.ship_counts(s1, water.intersection(port, 1), threshold=threshold, min_pixels=min_pixels)
counts_anchorage = constanta.ship_counts(s1, water.intersection(anchorage, 1), threshold=threshold, min_pixels=min_pixels)

# One table with a column per zone, plus the total
counts = pd.concat({'port': counts_port['n_ships'], 'anchorage': counts_anchorage['n_ships']}, axis=1)
counts['total'] = counts['port'] + counts['anchorage']

# Save detection counts to local data directory
counts.to_csv(os.path.join(data_dir, 'ship_counts.csv'))

## 7. Before and after the war

Plotting the time series of the count of detected ships. IMF Portwatch port calls are also plotted for validation purposes.

In [ ]:
from functions.plots import load_portwatch, monthly_means, plot_comparisons

# Download PortWatch port calls (or load them if the CSV already exists)
portwatch = load_portwatch('port260', os.path.join(data_dir, 'portwatch_constanta.csv'))

# Compare our ship detections with IMF PortWatch port calls
ships = pd.read_csv(os.path.join(data_dir, 'ship_counts.csv'), parse_dates=['date'])
monthly, corr = monthly_means(ships, portwatch, start_date, end_date)

events = {'Start of war': '2022-02-24', 'Russia leaves grain deal': '2023-07-17'}
fig = plot_comparisons(monthly, corr, events)
fig.show()

# Summary table: mean before and after the start of the war
war_start = events['Start of war']
portwatch = portwatch.loc[portwatch['date'] < end_date]

In [ ]:
# Summary table: mean before and after the start of the war
ships_before = ships[ships['date'] < war_start][['port', 'anchorage', 'total']].mean()
ships_after = ships[ships['date'] >= war_start][['port', 'anchorage', 'total']].mean()
calls_before = portwatch[portwatch['date'] < war_start][['portcalls']].mean()
calls_after = portwatch[portwatch['date'] >= war_start][['portcalls']].mean()

summary = pd.DataFrame({'before war': pd.concat([ships_before, calls_before]),
                        'after war': pd.concat([ships_after, calls_after])})
print(summary.round(1))

## 8. Timelapse showing vessel traffic change

Showcasing the change in vessels counted after the Russian Federation left the grain deal in 2023.

In [ ]:
# Timelapse of the detected ships around Russia leaving the grain deal (17 July 2023)
# https://ipython.readthedocs.io/en/stable/api/generated/IPython.display.html#IPython.display.HTML
# https://docs.python.org/3/library/base64.html#base64.b64encode
# https://developer.mozilla.org/en-US/docs/Web/URI/Reference/Schemes/data
import base64
from IPython.display import HTML, display

# Specify where to put gif file
gif_path = os.path.join(output_dir, 'timelapse_2023.gif')

# Running timelapse method for capturing the grain deal leave event (area = constanta.bbox)
constanta.make_timelapse(s1.filterDate('2023-04-01', '2023-11-01'), water, gif_path,
                         event_date='2023-07-17', method='threshold', threshold=threshold, min_pixels=min_pixels)

# Show the GIF in the notebook. Positron cannot display a GIF file directly, so we
# read the file as bytes, convert them to text (base64) and put that text in an HTML
# <img> tag. The browser decodes the text back into the image.
with open(gif_path, 'rb') as f:
    display(HTML(f'<img src="data:image/gif;base64,{base64.b64encode(f.read()).decode()}">'))

## 9. Limitations

**Detection**
1. **One fixed threshold**: We use a -15dB threshold for every image. The brightness of the sea changes depending on how rough the waves are. So on rough days sea pixels can be counted as ships, and on calm days less bright ships can be missed. We tried to solve this by using the CFAR detection and the adaptive threshold per image, but both did not work out in this limited time frame.
2. **Ships next to eachother**: In the port ships that are docked side by side can merge into one object, and one large ship can split into multiple. This amkes the port count less axact than the anchorage count.
3. **Small boats are missed**: An object needs at least 8 pixels to count, so the detection model mainly shows larger vessels
4. **No ground truth data**: We checked detections by eye, not against real ship positions at the time of each image, so we don't know exactly how many deteections are false or missed.

**Data**
1. **Fewer images between 2021 and 2024**: Sentinel-1B malfunctioned in December 2021, and Sentinel-1C only started in 2025. Those years have fewer images, so their counts are less certain.
2. **Snapshots, not images**: One image shows only the ships present at that moment, every 6 to 12 days. Ships that arrive and leave between two images are never seen.
3. **Changes in ESA processing**: ESA updated how Sentinel-1 images are processed during our period. This can shift brightness values slightly, so a fixed threshold may count differently in different years.
4. **Static water mask and simple zones**: The OSM water mask is one fixed shape, and port and anchorage are split at one longitude. Changes to the port over time, and ships near the dividing line, are not handled.

**Interpretation**
1. **Correlation, not cause**: We can see that counts changed around war events, but radar can't tell us why. Other things, like the economy, the season or the weather, can also change ship traffic.
2. **Different things compared with PortWatch.**: PortWatch counts ships arriving per day, while we count ships present in one image, including ships waiting at anchor. So we compare the trend, not the exact numbers.

## Conclusion


#### Research Objective: *To create a script that counts the number of vessels anchored at Constanta's maritime port, which captures the pre- and post- war traffic trends, whilst quantifying how well this script matches IMF PortWatch vessel port calls.*

#### SRQ1: *How can vessels be detected in the Constanta port area given boxed Sentinel-1 imagery using backscatter thresholding?* 
Vessels can be detected as bright objects on dark water: VH pixels brighter than -15 dB inside the water mask, grouped into objects of at least 8 pixels. This works well on the open sea at the anchorage. Inside the port it overcounts, because quays, cranes and ships moored side by side also create bright objects. More advanced methods (CFAR and an adaptive threshold) did not give better results for our area because of time and resource constraints.

#### SRQ2: *How did the number of detected vessels change after the start of the Russia-Ukraine war in February 2022?*
On average we detected **[X]** vessels per image before the war and **[Y]** after, a change of **[+/- Z %]**. The change was largest in the **[anchorage / port]**, which suggests **[more ships waiting / more ships being handled]**. Around Russia leaving the grain deal in July 2023, the counts **[rose / fell / stayed the same]**.

#### SRQ3: *How well do the Sentinel-1 vessel counts agree with IMF PortWatch port calls?*
The monthly counts follow PortWatch with a Spearman correlation of **ρ = [value]** for the total area (**[port value]** for the port and **[anchorage value]** for the anchorage). This means the trends agree **[well / moderately / weakly]**. The absolute numbers differ, because PortWatch counts arrivals and we count ships present at one moment.

**Overall**, free Sentinel-1 images can show **[how / whether]** shipping traffic in Constanța changed during the war. The method is best for comparing trends over time, not for counting the exact number of ships (see Limitations).
In this project we built a reproducible script that counts vessels in the port and anchorage of Constanța on Sentinel-1 radar images from 2019 to 2025, and compared the counts with IMF PortWatch port calls.


## With more time